# Session 2 — Relational thinking with a World Cup football SQL database

_More informations and reference:_
- [WorldCup GitHuB](https://github.com/jfjelstul/worldcup/tree/master)

## 1. Introduction

In this notebook, we will explore the World Cup database with SQL queries that lead naturally to informative visualizations.

The goal is to:
- practice `SELECT`, `JOIN`, `GROUP BY`, `ORDER BY`, and `COUNT`
- connect relational queries to visual thinking
- transform query outputs into clear plots
- discover non-obvious historical patterns in football data


### 1.1 Why start with football?
Football data is intuitive, structured, relational, and easy to reason about.  
This lets us focus on **SQL thinking**, which we will later transfer to **bioinformatics databases**, such as variant, sample, annotation, and metadata tables.

### 1.2 Research mindset
Even though the example dataset is playful, the real goal is not sports analytics.  
The real goal is to learn how to ask questions such as:

- *Which entities are stored separately?*
- *How are they connected?*
- *What must be joined?*
- *What should be aggregated?*
- *How do we move from a scientific question to a formal query?*


### 1.3 SQL in a modern research workflow

In practice, researchers rarely work with SQL in isolation. A typical workflow may include: Jupyter notebooks, SQLite, PostgreSQL, DBeaver/pgAdmin, VScode, AI tools, etc.

| Tool | Best use case |
|---|---|
| Jupyter | teaching, experimentation, narrative analysis |
| SQLite | simple local relational database |
| PostgreSQL | serious research infrastructure, bigger datasets |
| DBeaver | browsing tables, inspecting schemas, running ad hoc queries |
| VS Code | integrating SQL with code, notebooks, scripts, version control |

_____________

## 2. Connecting to the database and exploring the data

### 2.1 Connecting to the database

**First step:** establish a reliable connection to the database.  
We wrap the connection and query execution into a reusable function:
- connect to the database file  
- execute an SQL query  
- return the result as a pandas DataFrame  

**Second step:** inspect the database structure by listing all available tables. 




In [2]:
import pandas as pd
import sqlite3 as sql

# Connect to the SQLite database and fetch the names of all tables

def get_df_from_query(db_path, query):
    conn = sql.connect(db_path)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


db_path = 'worldcup.db'  # Replace with your database path
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = get_df_from_query(db_path, query)
print("Tables in the database:")
print(tables_df.head())




Tables in the database:
             name
0     tournaments
1  confederations
2           teams
3         players
4        managers


### 2.2 Inspect the tables

Before asking a complicated question, first inspect the raw tables.

How can we explore the database?

- Use a GUI tool:
  - DBeaver → https://dbeaver.io/
  - VS Code (Database Client extensions)
  - pgAdmin → https://www.pgadmin.org/

- Visualize the schema:
  - https://dbdiagram.io/ (requires DBML/DDL, not raw query output)
  - https://schemaspy.org/
  - https://erdplus.com/

- Use Python (see codes below):
  - pandas (`read_sql_query`) for quick inspection
  - build custom tools (e.g. schema extraction, visualization)

- Use AI-assisted tools:
  - text-to-SQL interfaces
  - query explanation / debugging with LLMs

At this stage, the goal is simple:
understand what data we have before writing complex queries.

#### 2.2.1 Inspection examples - __SQLite master__ (quick)

One of the simple way to explore the data: 
Inspect the schema (table definitions) using SQL

In [12]:
# Query the schema of the database
schema_query = "SELECT sql FROM sqlite_master WHERE type='table';"
schema_df = get_df_from_query(db_path, schema_query)
schema_df.head()

,sql
0,CREATE TABLE tournaments(\n tournament_id TEX...
1,CREATE TABLE confederations(\n confederation_...
2,"CREATE TABLE teams(\n team_id TEXT NOT NULL,\..."
3,CREATE TABLE players(\n player_id TEXT NOT NU...
4,CREATE TABLE managers(\n manager_id TEXT NOT ...


#### 2.2.2 Inspection examples - __Pandas python module__ (quick)

Preview actual data by loading a table into pandas

In [14]:
# Load a specific table into a DataFrame and inspect it
matches_query = "SELECT * FROM players LIMIT 5;"  # Replace 'matches' with your table name
matches_df = get_df_from_query(db_path, matches_query)
matches_df.head()

,player_id,family_name,given_name,birth_date,female,goal_keeper,defender,midfielder,forward,count_tournaments,list_tournaments,player_wikipedia_link
0,P-35894,A'Court,Alan,1934-09-30,0,0,0,0,1,1,1958,https://en.wikipedia.org/wiki/Alan_A%27Court
1,P-29915,Aarønes,Ann Kristin,1973-01-19,1,0,0,1,1,2,"1995, 1999",https://en.wikipedia.org/wiki/Ann_Kristin_Aar%...
2,P-03484,Aaronson,Brenden,2000-10-22,0,0,0,0,1,1,2022,https://en.wikipedia.org/wiki/Brenden_Aaronson
3,P-04189,Abadzhiev,Stefan,1934-07-03,0,0,0,0,1,1,1966,https://en.wikipedia.org/wiki/Stefan_Abadzhiev
4,P-03523,Abalo,Jean-Paul,1975-06-26,0,0,1,0,0,1,2006,https://en.wikipedia.org/wiki/Jean-Paul_Abalo


#### 2.2.3 Inspection examples - __Generating a schema visualization (DBML)__

- extract table and column information using SQL (`PRAGMA`)
- detect primary and foreign keys
- convert the schema into **DBML format** (Database Markup Language)

This format can be directly used in tools like:
https://dbdiagram.io/

This allows us to quickly generate a visual representation of the database structure.

In [4]:
import sqlite3 as sql
import pandas as pd

db_path = "worldcup.db"
conn = sql.connect(db_path)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';", conn
)["name"].tolist()

dbml = ""

for table in tables:
    cols = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    
    dbml += f"Table {table} {{\n"
    
    for _, row in cols.iterrows():
        col = row["name"]
        dtype = row["type"] if row["type"] else "text"
        
        constraints = []
        if row["pk"]:
            constraints.append("pk")
        
        constraint_str = f" [{', '.join(constraints)}]" if constraints else ""
        
        dbml += f"  {col} {dtype}{constraint_str}\n"
    
    dbml += "}\n\n"

# FK-k
for table in tables:
    fks = pd.read_sql_query(f"PRAGMA foreign_key_list({table});", conn)
    
    for _, fk in fks.iterrows():
        dbml += f"Ref: {table}.{fk['from']} > {fk['table']}.{fk['to']}\n"

conn.close()

print(dbml)

Table tournaments {
  tournament_id TEXT [pk]
  tournament_name TEXT
  year INTEGER
  start_date DATE
  end_date DATE
  host_country TEXT
  winner TEXT
  host_won BOOLEAN
  count_teams INTEGER
  group_stage BOOLEAN
  second_group_stage BOOLEAN
  final_round BOOLEAN
  round_of_16 BOOLEAN
  quarter_finals BOOLEAN
  semi_finals BOOLEAN
  third_place_match BOOLEAN
  final BOOLEAN
}

Table confederations {
  confederation_id TEXT [pk]
  confederation_name TEXT
  confederation_code TEXT
  confederation_wikipedia_link TEXT
}

Table teams {
  team_id TEXT [pk]
  team_name TEXT
  team_code TEXT
  mens_team BOOLEAN
  womens_team BOOLEAN
  federation_name TEXT
  region_name TEXT
  confederation_id TEXT
  mens_team_wikipedia_link TEXT
  womens_team_wikipedia_link TEXT
  federation_wikipedia_link TEXT
}

Table players {
  player_id TEXT [pk]
  family_name TEXT
  given_name TEXT
  birth_date DATE
  female BOOLEAN
  goal_keeper BOOLEAN
  defender BOOLEAN
  midfielder BOOLEAN
  forward BOOLEAN
  count

![](vb_schema_all.png)

### 2.3 Quering the database

#### 2.3.1 `SELECT` and `WHERE`

The most basic task is to retrieve rows from a table.

##### _Pattern_
```sql
SELECT column1, column2
FROM table_name
WHERE condition;
```

Use `*` only for exploration.  
In real work, it is usually better to explicitly list the columns you want.

<div class="alert alert-block alert-info">
<b>Question 1:</b> Which players were born after Jan 1st, 2004 and already participated in the World Cup?
</div>

In [16]:
query = '''SELECT given_name, family_name, birth_date
            FROM players
            WHERE birth_date > "2004-01-01" 
            AND birth_date != "not available"
            -- AND birth_date IS NOT NULL;'''

tables_df = get_df_from_query(db_path, query)
print("Players born after January 1, 2004. In total: ", len(tables_df))
tables_df.head()

Players born after January 1, 2004. In total:  6


,given_name,family_name,birth_date
0,Jewison,Bennette,2004-06-15
1,Bilal,El Khannous,2004-05-10
2,not applicable,Gavi,2004-08-05
3,Abdul Fatawu,Issahaku,2004-03-08
4,Garang,Kuol,2004-09-15


⚠️ Important: not all missing values are stored as NULL.

In this dataset, missing given_name are stored as the string "not applicable",
so `IS NOT NULL` does not filter them out.

#### 2.3.2 `JOIN`,`COUNT`,`GROUP BY` `ORDER BY` and `LIMIT`

Most interesting questions require both **combining data** and **summarizing it**.

For example:

> How many players does each team have?

To answer this:
- we need to **JOIN** tables to connect players to teams  
- then **GROUP BY** team to count players  

This pattern appears constantly in real-world data analysis:
- variants per gene  
- samples per condition  
- measurements per experiment  
- players per team  

In other words:
JOIN → adds context  
GROUP BY → extracts insight

<div class="alert alert-block alert-info">
<b>Question 2:</b> Which players scored the most goals in the World Cup? 
</div>

In [6]:
# Show the top 10 players with their team name, given name, family name, and number of goals scored. Order by number of goals scored in descending order.

query = '''SELECT t.team_name, p.given_name, p.family_name, COUNT(g.goal_id) AS goals
FROM players p
INNER JOIN goals g ON g.player_id = p.player_id
INNER JOIN teams t ON g.team_id = t.team_id
WHERE p.female = 0
GROUP BY t.team_id, p.given_name, p.family_name
ORDER BY COUNT(g.goal_id) DESC
LIMIT 10;'''
goal_counts_df = get_df_from_query(db_path, query)
goal_counts_df.head(10)

,team_name,given_name,family_name,goals
0,Germany,Miroslav,Klose,16
1,Brazil,not applicable,Ronaldo,15
2,West Germany,Gerd,Müller,14
3,Argentina,Lionel,Messi,13
4,France,Just,Fontaine,13
5,Brazil,not applicable,Pelé,12
6,France,Kylian,Mbappé,12
7,Hungary,Sándor,Kocsis,11
8,Argentina,Gabriel,Batistuta,10
9,England,Gary,Lineker,10


In [7]:
## Create a new column in the goal_counts_df DataFrame that combines the given_name and family_name columns into a single column called player_name. 
## If given_name is "not applicable", use only the family_name as the player name.

def player_full_name(row):
    if pd.isna(row['given_name']) or pd.isna(row['family_name']):
        return "Unknown Player"
    elif row['given_name'] == "not applicable":
        return row['family_name']
    return f"{row['given_name']} {row['family_name']}"

goal_counts_df['player_name'] = goal_counts_df.apply(player_full_name, axis=1)

## Make a barchart using plotly express

from matplotlib import markers
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

fig = px.bar(goal_counts_df, x="goals", y="player_name", orientation="h",
                    title="Top 10 Players with Most Goals in World Cup",
                    labels={"goals": "Number of Goals", "player_name": "Player Name"},
                    color="team_name")
fig.show()
    

#### 2.3.3 `DISTINCT`

<div class="alert alert-block alert-info">
<b>Question 3:</b> How many players has each national team had across all World Cups?
</div>

In [89]:
query = '''
        SELECT t.team_name, COUNT(DISTINCT(s.tournament_id)) AS n_tournament, COUNT(DISTINCT(p.player_id)) AS n_players
        FROM players p
        INNER JOIN squads s ON s.player_id = p.player_id
        INNER JOIN teams t ON s.team_id = t.team_id
        WHERE p.female = 0
        GROUP BY t.team_id
        ORDER BY COUNT(DISTINCT(p.player_id)) DESC
        LIMIT 10
        ;'''
team_counts_df = get_df_from_query(db_path, query)
team_counts_df.head(10)

,team_name,n_tournament,n_players
0,Brazil,22,345
1,Argentina,18,298
2,Italy,18,285
3,Mexico,17,267
4,Spain,16,265
5,France,16,257
6,England,16,249
7,Uruguay,14,236
8,Belgium,14,218
9,Sweden,12,216


In [90]:
### Make a barchart using plotly express of the teams with the most players in the World Cup and the number of tournaments they participated in, team name in y, and n_plaxyers in x horizontal bars

fig = px.bar(team_counts_df, x="n_players", y="team_name", orientation="h",
                 title="Top 10 Teams with Most Players in World Cup",
                 labels={"n_players": "Number of Players", "team_name": "Team Name"},
                 color="n_tournament")
fig.show()

<div class="alert alert-block alert-info">
<b>Question:</b> Which teams have scored the most goals in the history of the World Cup?
</div>

In [125]:
### Which teams have scored the most goals in the history of the World Cup?

sql_goals_by_team = """
SELECT t.team_name, COUNT(DISTINCT(g.goal_id)) AS n_goals
FROM goals g
JOIN teams t ON g.team_id = t.team_id
JOIN players p ON g.player_id = p.player_id
WHERE p.female = 0
GROUP BY t.team_id
ORDER BY n_goals DESC
LIMIT 20;
"""

df_goals_by_team = get_df_from_query(db_path, sql_goals_by_team)
df_goals_by_team.head(20)


fig = px.bar(df_goals_by_team, x="n_goals", y="team_name", orientation="h",
                    title="Top 20 Teams with Most Goals in World Cup History",
                    labels={"n_goals": "Number of Goals", "team_name": "Team Name"},
                    color="n_goals")
fig.update_layout(template="plotly_white")



#### 2.3.2 `CTE`, `CASE WHEN`, and `UNION ALL`

##### _Advanced questions_

<div class="alert alert-block alert-info">
<b>Question 4:</b> Which confederations became more dominant over time, and which remained marginal?
</div>

This question is more complex because it connects:
- tournaments,
- matches,
- teams,
- and confederations.

We will calculate match outcomes from the perspective of participating teams and summarize them by confederation and year.

In [45]:
sql_confederation_wins = """
WITH team_results AS (
    SELECT
        m.tournament_id,
        t.year,
        home.confederation_id,
        CASE
            WHEN m.home_team_win = 1 THEN 'Win'
            WHEN m.draw = 1 THEN 'Draw'
            WHEN m.away_team_win = 1 THEN 'Loss'
            ELSE 'Other'
        END AS result
    FROM matches m
    JOIN tournaments t
      ON m.tournament_id = t.tournament_id
    JOIN teams home
      ON m.home_team_id = home.team_id

    WHERE tournament_name LIKE '%Men%'

    UNION ALL

    SELECT
        m.tournament_id,
        t.year,
        away.confederation_id,
        CASE
            WHEN m.away_team_win = 1 THEN 'Win'
            WHEN m.draw = 1 THEN 'Draw'
            WHEN m.home_team_win = 1 THEN 'Loss'
            ELSE 'Other'
        END AS result
    FROM matches m
    JOIN tournaments t
      ON m.tournament_id = t.tournament_id
    JOIN teams away
      ON m.away_team_id = away.team_id
    WHERE tournament_name LIKE '%Men%'
)
SELECT
    tr.year,
    c.confederation_name,
    tr.result,
    COUNT(*) AS n_matches
FROM team_results tr
JOIN confederations c
  ON tr.confederation_id = c.confederation_id
WHERE tr.result IN ('Win', 'Draw', 'Loss')
GROUP BY tr.year, c.confederation_name, tr.result
ORDER BY tr.year, c.confederation_name, tr.result;
"""

df_confed = get_df_from_query(db_path, sql_confederation_wins)
df_confed.head()

,year,confederation_name,result,n_matches
0,1930,"Confederation of North, Central American and C...",Loss,4
1,1930,"Confederation of North, Central American and C...",Win,2
2,1930,South American Football Confederation,Loss,8
3,1930,South American Football Confederation,Win,12
4,1930,Union of European Football Associations,Loss,6


In [48]:
#### Vizualize the data using plotly express, creating a heatmap of wins by confederation and year, with a color scale that emphasizes the differences in the number of wins.

df_confed_wins = df_confed[df_confed["result"] == "Win"].copy()

apply_short_names = {"Union of European Football Associations" : "UEFA", "Confederation of African Football" : "CAF", "Confederation of North, Central American and Caribbean Association Football" : "CONCACAF", "Asian Football Confederation" : "AFC", "South American Football Confederation" : "CONMEBOL", "Oceania Football Confederation" : "OFC", "Confederation of Independent Football Associations" : "CONIFA"}
df_confed_wins["confederation_name"] = df_confed_wins["confederation_name"].replace(apply_short_names)

heatmap_data = df_confed_wins.pivot(
    index="confederation_name",
    columns="year",
    values="n_matches"
).fillna(0)

fig = px.imshow(
    heatmap_data,
    aspect="equal",
    title="World Cup wins by confederation and year",
    labels=dict(x="Year", y="Confederation", color="Wins"),
    color_continuous_scale="turbo"  # Turbo színskála beállítása
)
fig.update_layout(template="plotly_white")
fig.show()

### 2.4 Questions that are interesting to explore

**Penalty shootouts and rule changes**

In this section, we explore penalty shootout outcomes in FIFA World Cup tournaments between 1982 and 2022.

Our main question is whether the stricter enforcement of goalkeeper positioning rules, introduced in 2022, was associated with any visible change in penalty conversion success.

More specifically, we examine:

- penalty conversion rates before and after 2022,
- team-level penalty success patterns,
- and whether the available data suggest any clear trend.

> **Notes:**  
> - This is an exploratory analysis, not a causal proof.  
> - The post-2022 period currently contains only one World Cup tournament, so any conclusion should be interpreted with caution.
> - A significant rule change was implemented by IFAB in July 2022, strictly requiring goalkeepers to maintain at least one foot on the goal line until the ball is struck, monitored closely by Video Assistant Referees (VAR).

<div class="alert alert-block alert-info">
<b>Question 5:</b> Did penalty conversion rates change after the stricter goalkeeper-positioning rule enforcement introduced in 2022?
</div>

In [3]:
# Has the success rate of penalty kicks increased following the rule change regarding penalty kicks (requiring defenders to remain on the goal line)?

sql_penalty_kicks = """
SELECT tournament_id,
    CASE 
        WHEN CAST(substr(tournament_id, 4, 4) AS INT) < 2022 THEN 'Before 2022'
        ELSE  'After 2022'
    END AS period,
    COUNT(*) AS total_penalties,
    SUM(converted) AS successful_penalties,
    ROUND(AVG(converted) * 100, 2) AS success_rate_pct
FROM penalty_kicks
INNER JOIN players ON penalty_kicks.player_id = players.player_id
WHERE female = 0
GROUP BY period, tournament_id
ORDER BY period DESC; -- Hogy a sorrend logikus legyen
"""

df_penalties = get_df_from_query(db_path, sql_penalty_kicks)
df_penalties["success_rate"] = df_penalties["successful_penalties"] / df_penalties["total_penalties"]
df_penalties


,tournament_id,period,total_penalties,successful_penalties,success_rate_pct,success_rate
0,WC-1982,Before 2022,12,9,75.00,0.750000
1,WC-1986,Before 2022,27,21,77.78,0.777778
2,WC-1990,Before 2022,38,28,73.68,0.736842
3,WC-1994,Before 2022,29,18,62.07,0.620690
4,WC-1998,Before 2022,28,20,71.43,0.714286
5,WC-2002,Before 2022,19,13,68.42,0.684211
6,WC-2006,Before 2022,33,21,63.64,0.636364
7,WC-2010,Before 2022,18,14,77.78,0.777778
8,WC-2014,Before 2022,36,26,72.22,0.722222
9,WC-2018,Before 2022,39,26,66.67,0.666667


In [10]:
fig = px.bar(
    df_penalties,
    x="period",
    y="success_rate_pct",
    title="Penalty Kick Success Rate Before and After 2022 Rule Change",
    labels={"period": "Period", "success_rate_pct": "Success Rate (%)"},
    color="period",
    text="success_rate_pct"
)

fig.update_layout(template="plotly_white")
fig.show()

<div class="alert alert-block alert-info">
<b>Question 6:</b> Do some teams show consistently different penalty success rates depending on home or away status?
</div>

In [ ]:
sql_penalty_kicks = """ 
SELECT team_name,
    CASE 
        WHEN home_team = 1 THEN 'Home status'
        ELSE 'Away status' 
    END AS team_status,
    COUNT(*) AS total_penalties,
    SUM(converted) AS successful_penalties,
    ROUND(AVG(converted) * 100, 2) AS success_rate_pct
FROM penalty_kicks
INNER JOIN teams ON penalty_kicks.team_id = teams.team_id
GROUP BY teams.team_id, team_status;
"""
df_penalties = get_df_from_query(db_path, sql_penalty_kicks)
df_penalties.head()

,team_name,team_status,total_penalties,successful_penalties,success_rate_pct
0,Argentina,Hazai státusz,18,15,83.33
1,Argentina,Vendég státusz,13,10,76.92
2,Australia,Vendég státusz,3,1,33.33
3,Belgium,Vendég státusz,5,5,100.00
4,Brazil,Hazai státusz,28,21,75.00


> **Note:**  
> This team-level view is more difficult to interpret because many teams have only a small number of penalties.  
> Large percentage differences may therefore reflect small sample sizes rather than stable team characteristics.

In [39]:
import plotly.express as px             

fig = px.bar(df_penalties, x="team_name", y="success_rate_pct", color="team_status",
                    title="Penalty Kick Success Rate by Team Status",
                    labels={"success_rate_pct": "Success Rate (%)", "team_name": "Team Name", "team_status": "Team Status"},
                    barmode="group")  # "group" for side-by-side bars, "stack" for stacked bars
fig.update_layout(template="plotly_white", xaxis_tickangle=-45)
fig.show()